# ICT Market Structure — Demo
> `strategies/market_structure.py` — Swing detection · BOS · CHoCH · Order Blocks

In [3]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")

import yfinance as yf
import pandas as pd
from strategies.market_structure import MarketStructure, analyse

In [4]:
import importlib
import strategies.market_structure as ms_module
importlib.reload(ms_module)
from strategies.market_structure import MarketStructure, analyse

## 1 · Load EURUSD 1H data

In [5]:
raw = yf.download('EURUSD=X', period='60d', interval='1h', auto_adjust=True)

# yfinance returns MultiIndex columns — flatten to single level
df = raw.copy()
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0).str.lower()
else:
    df.columns = df.columns.str.lower()
df.index = pd.to_datetime(df.index)
df = df.dropna()

print(f'Rows: {len(df):,}  |  {df.index[0]} → {df.index[-1]}')
df.tail(3)

[*********************100%***********************]  1 of 1 completed

Rows: 1,403  |  2026-02-04 00:00:00+00:00 → 2026-04-28 02:00:00+00:00


Price,close,high,low,open,volume
Datetime,,,,,
2026-04-28 00:00:00+00:00,1.172196,1.173020,1.172058,1.172745,0
2026-04-28 01:00:00+00:00,1.171646,1.172196,1.171646,1.172058,0
2026-04-28 02:00:00+00:00,1.171646,1.171646,1.171509,1.171646,0


## 2 · Run analysis

In [6]:
ms = MarketStructure(df, swing_lookback=5)

print(f'Swing Highs : {len(ms.swing_highs)}')
print(f'Swing Lows  : {len(ms.swing_lows)}')
print(f'BOS/CHoCH   : {len(ms.structure_events)}')
print(f'Order Blocks: {len(ms.order_blocks)}')

Swing Highs : 109
Swing Lows  : 105
BOS/CHoCH   : 55
Order Blocks: 55


## 3 · Interactive chart (last 300 candles)

In [7]:
fig = ms.plot(
    last_n=300,
    title='EURUSD 1H — ICT Market Structure',
    height=820,
    max_events=10,           # show only 10 most recent BOS/CHoCH
    show_mitigated_ob=False, # hide already-visited OBs
)
fig.show()

## 4 · Events summary table

In [8]:
ms.summary().tail(15)

,timestamp,event,price,broken_level
40,2026-04-06 06:00:00+00:00,CHoCH_bull,1.153669,1.153270
41,2026-04-07 01:00:00+00:00,CHoCH_bear,1.153270,1.153802
42,2026-04-07 17:00:00+00:00,CHoCH_bull,1.158078,1.157809
43,2026-04-08 12:00:00+00:00,BOS_bull,1.171783,1.171097
44,2026-04-10 06:00:00+00:00,CHoCH_bear,1.168770,1.168907
45,2026-04-10 13:00:00+00:00,CHoCH_bull,1.173020,1.172608
46,2026-04-13 13:00:00+00:00,BOS_bull,1.170686,1.170275
47,2026-04-16 02:00:00+00:00,BOS_bull,1.182033,1.181056
48,2026-04-17 08:00:00+00:00,BOS_bull,1.179802,1.179106
49,2026-04-17 19:00:00+00:00,CHoCH_bear,1.177163,1.177718


## 5 · Order blocks table

In [9]:
ms.order_blocks_df()

,timestamp,kind,high,low,open,close,mitigated,mitigated_at
0,2026-02-05 00:00:00+00:00,bearish,1.180777,1.180359,1.180498,1.180638,True,2026-02-05 01:00:00+00:00
1,2026-02-05 19:00:00+00:00,bearish,1.180080,1.179663,1.179663,1.179802,True,2026-02-05 20:00:00+00:00
2,2026-02-06 11:00:00+00:00,bullish,1.179663,1.178967,1.179523,1.179384,True,2026-02-06 12:00:00+00:00
3,2026-02-06 21:00:00+00:00,bullish,1.182732,1.181754,1.182592,1.182173,True,2026-02-06 22:00:00+00:00
4,2026-02-10 09:00:00+00:00,bearish,1.192321,1.191185,1.191327,1.191753,True,2026-02-10 10:00:00+00:00
5,2026-02-13 02:00:00+00:00,bearish,1.187507,1.186662,1.186944,1.187225,True,2026-02-13 03:00:00+00:00
6,2026-02-16 10:00:00+00:00,bearish,1.187085,1.186662,1.186803,1.187085,True,2026-02-16 11:00:00+00:00
7,2026-02-17 19:00:00+00:00,bullish,1.185396,1.184694,1.185255,1.185115,True,2026-02-17 20:00:00+00:00
8,2026-02-18 15:00:00+00:00,bearish,1.182872,1.181893,1.181893,1.182592,True,2026-02-18 16:00:00+00:00
9,2026-02-19 07:00:00+00:00,bearish,1.180638,1.179663,1.179941,1.180498,True,2026-02-19 08:00:00+00:00


## 6 · 4H chart
Higher-timeframe context for confluence.

In [12]:
raw_4h = yf.download('EURUSD=X', period='120d', interval='4h', auto_adjust=True)
df_4h = raw_4h.copy()
if isinstance(df_4h.columns, pd.MultiIndex):
    df_4h.columns = df_4h.columns.get_level_values(0).str.lower()
else:
    df_4h.columns = df_4h.columns.str.lower()
df_4h.index = pd.to_datetime(df_4h.index)
df_4h = df_4h.dropna()

ms_4h = MarketStructure(df_4h, swing_lookback=5)
ms_4h.plot(
    last_n=200,
    title='EURUSD 4H — ICT Market Structure',
    height=820,
).show()

[*********************100%***********************]  1 of 1 completed


In [1]:
import sys
sys.path.insert(0, r"E:\Claude Projects\quant_ict_trader")
import yfinance as yf
import pandas as pd

In [2]:
raw = yf.download('EURUSD=X', period='60d', interval='1h', auto_adjust=True)
df = raw.copy()
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0).str.lower()
df.index = pd.to_datetime(df.index)
df = df.dropna()

raw_4h = yf.download('EURUSD=X', period='120d', interval='4h', auto_adjust=True)
df_4h = raw_4h.copy()
if isinstance(df_4h.columns, pd.MultiIndex):
    df_4h.columns = df_4h.columns.get_level_values(0).str.lower()
df_4h.index = pd.to_datetime(df_4h.index)
df_4h = df_4h.dropna()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [ ]:
from strategies.entry_model import EntryModel

model = EntryModel(
    df_high=df_4h,
    df_low=df,
    instrument="EURUSD",
    account_balance=10000,
    risk_pct=0.01,
    daily_loss_limit=0.03,
    max_drawdown=0.10,
    tp1_rr=1.0,
    tp2_rr=2.0,
    sl_buffer_pips=3.0,
    min_rr=1.5,
)
print(f"Signals: {len(model.signals)}")
print(model.latest_signal())

In [4]:
from utils.signal_viewer import plot_all_signals
from strategies.market_structure import MarketStructure

ms = MarketStructure(df, swing_lookback=5)
ms_fig = ms.plot(last_n=300, title="EURUSD 1H — ICT Market Structure")
ms_fig.write_html(r"E:\Claude Projects\quant_ict_trader\chart.html")

plot_all_signals(
    df=df,
    signals=model.signals,
    candles_before=60,
    candles_after=20,
    save_html=r"E:\Claude Projects\quant_ict_trader\signals.html"
)

print("Done — open chart.html and signals.html in browser")

Saved 6 signal charts → E:\Claude Projects\quant_ict_trader\signals.html
Done — open chart.html and signals.html in browser


In [ ]:
import importlib
import main as main_module
importlib.reload(main_module)
from main import run_once, get_sheet, SHEET_HEADERS

ws = get_sheet()
ws.clear()
ws.append_row(SHEET_HEADERS)
run_once(ws, set())

2026-04-29 08:48:54,247  INFO  ── Run at 2026-04-29 08:48 ──────────────────────────
2026-04-29 08:48:54,247  INFO  Processing EURUSD...
2026-04-29 08:48:55,239  INFO    EURUSD — 1H: 1404 rows  4H: 706 rows
2026-04-29 08:48:55,346  INFO    EURUSD — Signals: 6
2026-04-29 08:48:55,346  INFO  
  SHORT EURUSD
  Entry  : 1.17385
  SL     : 1.17608  (22.3 pips)
  TP1    : 1.17161  (RR 1.0:1)
  TP2    : 1.16825  (RR 2.5:1)
  Size   : 0.64 lots
  Risk   : $100.00 (1.0%)
  Trend  : BOS_bear on HTF
  Zone   : Bearish FVG
  Confirm: Bear liquidity grab at 1.17165
2026-04-29 08:48:55,731  INFO  Logged to Sheets: SHORT EURUSD @ 1.19161
2026-04-29 08:48:56,249  INFO  Logged to Sheets: SHORT EURUSD @ 1.1909
2026-04-29 08:48:56,698  INFO  Logged to Sheets: SHORT EURUSD @ 1.18231
2026-04-29 08:48:57,066  INFO  Logged to Sheets: SHORT EURUSD @ 1.1812
2026-04-29 08:48:57,565  INFO  Logged to Sheets: SHORT EURUSD @ 1.18022
2026-04-29 08:48:57,949  INFO  Logged to Sheets: SHORT EURUSD @ 1.17385
Saved 6 sig

{'2026-04-29 03:00:00+00:00'}